# EPIC Clarity Device Exposure Hydration

This notebook hydrates the OMOP DEVICE_EXPOSURE table from EPIC Clarity OR implant data.

## Source Table
- `_exponent._bronze_epic_clarity_*.dbo_OR_IMP`
- `_exponent._bronze_epic_clarity_*.dbo_OR_LOG` (for patient/encounter linkage)

## OMOP Required Fields
- device_exposure_id (surrogate key from mapping)
- person_id (via encounter/patient linkage)
- device_concept_id (defaulting to 0 if no mapping)
- device_exposure_start_date (from OR log or implant date)

In [ ]:
source = 'epic_clarity'

In [ ]:
silver_device_exposure_df = spark.sql(f'''
SELECT
    CONCAT_WS(CHR(31), 'epic_clarity', 'OR_IMP', 'IMPLANT_ID', oi.IMPLANT_ID) AS device_exposure_source_value,
    CONCAT_WS(CHR(31), 'epic_clarity', 'OR_IMP', 'IMPLANT_ID', oi.IMPLANT_ID) AS device_source_value,
    oi.IMPLANT_NAME AS device_name,
    oi.STATUS_C AS device_type_source_value,
    CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', oi.PAT_ID) AS person_source_value,
    -- Use RECEIVED_DATE as device exposure start date
    COALESCE(CAST(oi.RECEIVED_DATE AS DATE), CURRENT_DATE()) AS device_exposure_start_date,
    -- UDI for concept mapping
    oi.STATIC_UDI AS unique_device_id,
    CURRENT_TIMESTAMP() AS updated_tsp
FROM _exponent._bronze_epic_clarity.or_imp oi
WHERE oi.IMPLANT_ID IS NOT NULL
  AND oi.PAT_ID IS NOT NULL
''')

display(silver_device_exposure_df)
silver_device_exposure_df.createOrReplaceTempView("silver_device_exposure")

In [ ]:
%sql
MERGE INTO _exponent.omop_silver.device_exposure AS target
USING silver_device_exposure AS source
ON target.device_exposure_source_value = source.device_exposure_source_value

WHEN MATCHED AND NOT (
    target.device_source_value <=> source.device_source_value
    AND target.device_exposure_start_date <=> source.device_exposure_start_date
    AND target.unique_device_id <=> source.unique_device_id
)
THEN UPDATE SET
    target.device_source_value = source.device_source_value,
    target.device_exposure_start_date = source.device_exposure_start_date,
    target.unique_device_id = source.unique_device_id,
    target.last_mod_tsp = source.updated_tsp

WHEN NOT MATCHED THEN INSERT (
    device_exposure_source_value,
    device_source_value,
    device_exposure_start_date,
    unique_device_id,
    last_mod_tsp
)
VALUES (
    source.device_exposure_source_value,
    source.device_source_value,
    source.device_exposure_start_date,
    source.unique_device_id,
    source.updated_tsp
)

In [ ]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_device_exposure (
    source_system,
    device_exposure_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.device_exposure_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    COALESCE(s.last_mod_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT 'epic_clarity' AS source_system, device_exposure_source_value, last_mod_tsp
    FROM _exponent.omop_silver.device_exposure
    WHERE device_exposure_source_value IS NOT NULL
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_device_exposure x
    ON s.device_exposure_source_value = x.device_exposure_source_value
    AND x.source_system = 'epic_clarity'

In [ ]:
gold_device_exposure_df = spark.sql("""
SELECT
    m.device_exposure_id,
    mp.person_id,
    0 AS device_concept_id,
    s.device_exposure_start_date,
    s.device_exposure_start_date AS device_exposure_start_datetime,
    s.device_source_value,
    s.unique_device_id,
    silver.person_source_value
FROM _exponent.omop_silver.device_exposure s
INNER JOIN _exponent.omop_mapping.source_to_device_exposure m
    ON s.device_exposure_source_value = m.device_exposure_source_value
    AND m.source_system = 'epic_clarity'
    AND m.active_flag = TRUE
INNER JOIN silver_device_exposure silver
    ON s.device_exposure_source_value = silver.device_exposure_source_value
LEFT JOIN _exponent.omop_mapping.source_to_person mp
    ON silver.person_source_value = mp.person_source_value
    AND mp.source_system = 'epic_clarity'
    AND mp.active_flag = TRUE
WHERE mp.person_id IS NOT NULL
""")

display(gold_device_exposure_df)
gold_device_exposure_df.createOrReplaceTempView("gold_device_exposure")

In [ ]:
%sql
MERGE INTO _exponent.omop.device_exposure AS target
USING gold_device_exposure AS source
ON target.device_exposure_id = source.device_exposure_id

WHEN MATCHED AND NOT (
    target.person_id <=> source.person_id
    AND target.device_concept_id <=> source.device_concept_id
    AND target.device_exposure_start_date <=> source.device_exposure_start_date
    AND target.device_source_value <=> source.device_source_value
    AND target.unique_device_id <=> source.unique_device_id
)
THEN UPDATE SET
    target.person_id = source.person_id,
    target.device_concept_id = source.device_concept_id,
    target.device_exposure_start_date = source.device_exposure_start_date,
    target.device_exposure_start_datetime = source.device_exposure_start_datetime,
    target.device_source_value = source.device_source_value,
    target.unique_device_id = source.unique_device_id

WHEN NOT MATCHED THEN INSERT (
    device_exposure_id,
    person_id,
    device_concept_id,
    device_exposure_start_date,
    device_exposure_start_datetime,
    device_source_value,
    unique_device_id
)
VALUES (
    source.device_exposure_id,
    source.person_id,
    source.device_concept_id,
    source.device_exposure_start_date,
    source.device_exposure_start_datetime,
    source.device_source_value,
    source.unique_device_id
)